# PDF to Structured Data — ETL Pipeline

End-to-end: raw PDF in, queryable database out.

Three document types. One pipeline. No fixed schema.

---

**Stages**
1. Detect — classify each page as native or image-based
2. Extract — pdfplumber for native, Claude vision for scanned
3. Structure — LLM infers schema and returns typed JSON
4. Load — insert into DuckDB and query

Set your document paths and API key in Stage 0, then run all cells.

In [ ]:
import os
import json
import base64
import tempfile
import io
from pathlib import Path

import pdfplumber
import duckdb
import anthropic
from pdf2image import convert_from_path
from PIL import Image
from IPython.display import display, Markdown

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY not set.\n"
        "Run: export ANTHROPIC_API_KEY=your_key_here"
    )

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Document paths — swap these for your own PDFs
DOCUMENTS = {
    "exam_paper":     "sample_pdfs/exam_paper.pdf",
    "invoice":        "sample_pdfs/invoice.pdf",
    "research_paper": "sample_pdfs/research_paper.pdf",
}

# Pipeline settings
NATIVE_CHAR_THRESHOLD = 100   # pages below this char count → image path
VISION_DPI            = 200   # render resolution for scanned pages
MAX_PAGES             = 30    # cap for large documents
DB_PATH               = "output/pipeline.db"

Path("output").mkdir(exist_ok=True)
print("Configuration loaded.")
print(f"Documents: {list(DOCUMENTS.keys())}")

---
## Stage 1 — Detect

Classify each page as `native` (text-selectable) or `image` (scanned).

pdfplumber attempts text extraction on every page. Pages returning fewer
than `NATIVE_CHAR_THRESHOLD` characters are flagged for the vision path.
This handles three real-world cases: fully native PDFs, fully scanned PDFs,
and mixed documents where some pages are native and others are not.

In [ ]:
def detect_pages(pdf_path: str) -> dict:
    """
    Classify each page of a PDF as native or image-based.
    Returns a detection result with per-page classification
    and a top-level summary type.
    """
    page_types = {}

    with pdfplumber.open(pdf_path) as pdf:
        pages = pdf.pages[:MAX_PAGES]
        for i, page in enumerate(pages, start=1):
            text = page.extract_text() or ""
            char_count = len(text.strip())
            page_types[i] = {
                "type":       "native" if char_count >= NATIVE_CHAR_THRESHOLD else "image",
                "char_count": char_count,
            }

    types = {p["type"] for p in page_types.values()}
    if types == {"native"}:
        pdf_type = "native"
    elif types == {"image"}:
        pdf_type = "image"
    else:
        pdf_type = "mixed"

    return {
        "pdf_path": pdf_path,
        "pdf_type": pdf_type,
        "total_pages": len(page_types),
        "native_pages": sum(1 for p in page_types.values() if p["type"] == "native"),
        "image_pages":  sum(1 for p in page_types.values() if p["type"] == "image"),
        "pages": page_types,
    }


# Run detection on all documents
detections = {}
for name, path in DOCUMENTS.items():
    if not Path(path).exists():
        print(f"  Skipping {name} — file not found at {path}")
        continue
    result = detect_pages(path)
    detections[name] = result
    print(
        f"  {name:<16} {result['pdf_type']:<8} "
        f"{result['total_pages']} pages  "
        f"({result['native_pages']} native, {result['image_pages']} image)"
    )

---
## Stage 2 — Extract

Two paths depending on page classification:

**Native pages** — pdfplumber extracts text directly, preserving layout
structure and character-level positioning. Fast and deterministic.

**Image pages** — pages are rendered to JPEG at 200 DPI using pdf2image,
then passed to Claude vision. The model reads the image and returns the
text content with layout cues. OCR and structure understanding happen in
one API call — no separate OCR step.

Mixed documents are handled page by page. The output is always the same
structure regardless of path: a dict of `{page_number: text}`.

In [ ]:
def extract_native_pages(pdf_path: str, page_types: dict) -> dict:
    """Extract text from native pages using pdfplumber."""
    text_by_page = {}
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages[:MAX_PAGES], start=1):
            if page_types.get(i, {}).get("type") == "native":
                text_by_page[i] = (page.extract_text() or "").strip()
    return text_by_page


def render_page_to_base64(image: Image.Image) -> str:
    """Convert a PIL image to base64 JPEG for the Claude API."""
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=85)
    return base64.standard_b64encode(buffer.getvalue()).decode("utf-8")


def extract_image_pages(pdf_path: str, page_types: dict) -> dict:
    """
    Extract text from image-based pages using Claude vision.
    Pages are rendered at VISION_DPI and passed as base64 JPEG.
    """
    image_page_nums = [
        num for num, info in page_types.items()
        if info["type"] == "image"
    ]
    if not image_page_nums:
        return {}

    images = convert_from_path(
        pdf_path,
        dpi=VISION_DPI,
        first_page=min(image_page_nums),
        last_page=min(max(image_page_nums), MAX_PAGES),
    )

    text_by_page = {}
    for i, image in zip(
        range(min(image_page_nums), min(max(image_page_nums), MAX_PAGES) + 1),
        images
    ):
        if i not in image_page_nums:
            continue

        response = client.messages.create(
            model="claude-opus-4-6",
            max_tokens=2000,
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type":       "base64",
                            "media_type": "image/jpeg",
                            "data":       render_page_to_base64(image),
                        }
                    },
                    {
                        "type": "text",
                        "text": (
                            "Extract all text from this page exactly as it appears. "
                            "Preserve structure, labels, and numerical values. "
                            "Return plain text only."
                        )
                    }
                ]
            }]
        )
        text_by_page[i] = response.content[0].text.strip()

    return text_by_page


def extract_document(detection: dict) -> dict:
    """
    Run the correct extraction path for each page.
    Returns unified text_by_page regardless of document type.
    """
    pdf_path   = detection["pdf_path"]
    page_types = detection["pages"]

    native_text = extract_native_pages(pdf_path, page_types)
    image_text  = extract_image_pages(pdf_path, page_types)

    text_by_page = {**native_text, **image_text}

    full_text = ""
    for page_num in sorted(text_by_page.keys()):
        full_text += f"\n--- PAGE {page_num} ---\n{text_by_page[page_num]}"

    return {
        "text_by_page": text_by_page,
        "full_text":    full_text.strip(),
        "page_count":   len(text_by_page),
    }


# Run extraction
extractions = {}
for name, detection in detections.items():
    print(f"Extracting {name}...")
    result = extract_document(detection)
    extractions[name] = result
    total_chars = sum(len(t) for t in result["text_by_page"].values())
    print(f"  {result['page_count']} pages, {total_chars:,} characters extracted")

---
## Stage 3 — Structure

The extracted text is passed to Claude with a two-part prompt:

1. **Characterise** — what type of document is this, and what fields does it contain?
2. **Extract** — return a typed JSON object with every field found

No schema is defined in advance. The model reads the document, decides
what fields exist, assigns types, and returns structured JSON. This is the
core design choice of the pipeline — schema inference rather than schema
enforcement.

The response is validated before passing to Stage 4. If a field is missing
or a type is wrong, the validation step flags it rather than silently
passing bad data to the database.

In [ ]:
def build_extraction_prompt(full_text: str) -> str:
    return f"""You are a data extraction pipeline. You will receive the text content
of a document and must return a structured JSON object.

Step 1 — Identify the document type and all fields present.
Step 2 — Return a single JSON object with this envelope:

{{
  "document_type": "invoice | exam_paper | research_paper | other",
  "confidence":    "high | medium | low",
  "fields":        {{ ...every field you can find, typed appropriately... }},
  "line_items":    [ ...array if the document contains tabular rows, else null... ],
  "metadata": {{
    "title":    "document title or null",
    "date":     "YYYY-MM-DD or null",
    "language": "ISO 639-1 code e.g. en"
  }}
}}

Rules:
- Use snake_case for all field names.
- Numeric values must be numbers, not strings.
- Dates must be YYYY-MM-DD strings.
- Do not invent fields that are not in the document.
- If a field is present but unreadable, use null.
- Return ONLY the JSON object. No explanation, no markdown fences.

DOCUMENT TEXT:
{full_text[:8000]}
"""


def extract_structure(full_text: str) -> dict:
    """Call Claude to infer schema and extract structured data."""
    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=3000,
        messages=[{
            "role":    "user",
            "content": build_extraction_prompt(full_text),
        }]
    )

    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    return json.loads(raw.strip())


def validate_structure(structured: dict) -> list:
    """
    Basic validation before load.
    Returns a list of warnings — empty list means clean.
    """
    warnings = []

    if "document_type" not in structured:
        warnings.append("Missing document_type")
    if "fields" not in structured or not isinstance(structured["fields"], dict):
        warnings.append("Missing or malformed fields dict")
    if structured.get("confidence") == "low":
        warnings.append("Low confidence extraction — review before use")
    if not structured.get("fields"):
        warnings.append("No fields extracted — document may be empty or unreadable")

    return warnings


# Run structuring
structured_data = {}
for name, extraction in extractions.items():
    print(f"Structuring {name}...")
    structured = extract_structure(extraction["full_text"])
    warnings   = validate_structure(structured)

    structured_data[name] = structured

    print(f"  Type       : {structured.get('document_type')}")
    print(f"  Confidence : {structured.get('confidence')}")
    print(f"  Fields     : {list(structured.get('fields', {}).keys())}")
    if warnings:
        for w in warnings:
            print(f"  ⚠  {w}")
    print()

    # Save JSON artefact
    out_path = f"output/{name}.json"
    with open(out_path, "w") as f:
        json.dump(structured, f, indent=2)
    print(f"  Saved → {out_path}")

---
## Stage 4 — Load

Structured JSON is inserted into DuckDB.

Two tables per run:
- `documents` — one row per document, top-level fields flattened
- `line_items` — one row per line item where applicable (invoices, question lists)

The schema is generated from the extracted fields — not defined in advance.
DuckDB's flexible column typing handles mixed-type fields gracefully.

A `schema.sql` file is written alongside the database for portability —
you can run the same schema on Postgres or BigQuery without changes to
the query logic.

In [ ]:
def create_schema(conn: duckdb.DuckDBPyConnection):
    """Create the universal document tables."""
    conn.execute("""
        CREATE TABLE IF NOT EXISTS documents (
            id            VARCHAR PRIMARY KEY,
            document_type VARCHAR,
            confidence    VARCHAR,
            title         VARCHAR,
            date          DATE,
            language      VARCHAR,
            fields        JSON,
            source_file   VARCHAR,
            loaded_at     TIMESTAMP DEFAULT current_timestamp
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS line_items (
            id            INTEGER,
            document_id   VARCHAR REFERENCES documents(id),
            position      INTEGER,
            data          JSON
        )
    """)


def insert_document(
    conn: duckdb.DuckDBPyConnection,
    name: str,
    structured: dict,
):
    """Insert one structured document into DuckDB."""
    import uuid
    doc_id   = str(uuid.uuid4())
    metadata = structured.get("metadata", {})
    fields   = structured.get("fields", {})

    # Parse date safely
    raw_date = metadata.get("date")
    doc_date = None
    if raw_date:
        try:
            from datetime import date
            doc_date = date.fromisoformat(raw_date)
        except Exception:
            pass

    conn.execute("""
        INSERT OR IGNORE INTO documents
            (id, document_type, confidence, title, date, language, fields, source_file)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, [
        doc_id,
        structured.get("document_type"),
        structured.get("confidence"),
        metadata.get("title"),
        doc_date,
        metadata.get("language"),
        json.dumps(fields),
        name,
    ])

    line_items = structured.get("line_items") or []
    for i, item in enumerate(line_items):
        conn.execute("""
            INSERT INTO line_items (id, document_id, position, data)
            VALUES (?, ?, ?, ?)
        """, [i, doc_id, i, json.dumps(item)])

    return doc_id


# Load into DuckDB
conn = duckdb.connect(DB_PATH)
create_schema(conn)

doc_ids = {}
for name, structured in structured_data.items():
    doc_id = insert_document(conn, name, structured)
    doc_ids[name] = doc_id
    print(f"  Loaded {name} → {doc_id[:8]}…")

conn.commit()
print(f"\nDatabase written to {DB_PATH}")

# Write schema.sql artefact
schema_sql = """
CREATE TABLE IF NOT EXISTS documents (
    id            VARCHAR PRIMARY KEY,
    document_type VARCHAR,
    confidence    VARCHAR,
    title         VARCHAR,
    date          DATE,
    language      VARCHAR,
    fields        JSON,
    source_file   VARCHAR,
    loaded_at     TIMESTAMP DEFAULT current_timestamp
);

CREATE TABLE IF NOT EXISTS line_items (
    id          INTEGER,
    document_id VARCHAR REFERENCES documents(id),
    position    INTEGER,
    data        JSON
);
"""
with open("output/schema.sql", "w") as f:
    f.write(schema_sql.strip())
print("Schema written to output/schema.sql")

---
## Query

The database is populated. Run any SQL against it.

Below are three queries — one per document type — to verify the extraction
and demonstrate that the structured data is immediately usable.

In [ ]:
# Overview of all loaded documents
result = conn.execute("""
    SELECT
        source_file,
        document_type,
        confidence,
        title,
        date,
        language
    FROM documents
    ORDER BY loaded_at
""").fetchdf()

display(result)

In [ ]:
# Extract specific fields from the invoice using JSON accessors
result = conn.execute("""
    SELECT
        json_extract_string(fields, '$.vendor_name')    AS vendor,
        json_extract_string(fields, '$.invoice_number') AS invoice_number,
        json_extract(fields, '$.total_amount')          AS total,
        json_extract_string(fields, '$.currency')       AS currency,
        date
    FROM documents
    WHERE document_type = 'invoice'
""").fetchdf()

display(result)

In [ ]:
# Line items joined to their parent document
result = conn.execute("""
    SELECT
        d.source_file,
        li.position + 1                              AS item_number,
        json_extract_string(li.data, '$.description') AS description,
        json_extract(li.data, '$.quantity')            AS qty,
        json_extract(li.data, '$.unit_price')          AS unit_price,
        json_extract(li.data, '$.amount')              AS amount
    FROM line_items li
    JOIN documents d ON li.document_id = d.id
    ORDER BY d.source_file, li.position
""").fetchdf()

display(result)

In [ ]:
# Line items joined to their parent document
result = conn.execute("""
    SELECT
        d.source_file,
        li.position + 1                              AS item_number,
        json_extract_string(li.data, '$.description') AS description,
        json_extract(li.data, '$.quantity')            AS qty,
        json_extract(li.data, '$.unit_price')          AS unit_price,
        json_extract(li.data, '$.amount')              AS amount
    FROM line_items li
    JOIN documents d ON li.document_id = d.id
    ORDER BY d.source_file, li.position
""").fetchdf()

display(result)

---
## Output artefacts

Three files are written to `output/` after a full pipeline run:

| File | Contents |
|------|----------|
| `pipeline.db` | DuckDB database — query with any SQL client |
| `schema.sql` | Table definitions — portable to Postgres, BigQuery |
| `{name}.json` | Raw structured JSON per document — inspect or reprocess |

To open the database in a SQL client:
```bash
duckdb output/pipeline.db
```

To run the schema on Postgres:
```bash
psql $DATABASE_URL < output/schema.sql
```

---

## Notes on failures

**Low confidence extractions** are flagged at Stage 3 but not dropped.
Inspect the raw JSON in `output/{name}.json` and re-run Stage 3 with
a more targeted prompt if needed.

**Missing fields** in the query results mean the model did not find that
field in the document — not that the extraction failed. Check the raw
`fields` JSON column to see what was actually extracted.

**Image-heavy pages** with complex multi-column layouts (common in
research papers) may produce partial extractions. The pipeline surfaces
this via the `confidence` field rather than silently passing bad data.

In [ ]:
conn.close()
print("Pipeline complete.")
print(f"  Database : {DB_PATH}")
print(f"  Schema   : output/schema.sql")
for name in structured_data:
    print(f"  JSON     : output/{name}.json")